# Portfolio sintetico: baseline classica con SCIP

Analisi degli artifact persistiti, senza eseguire la pipeline nel notebook. Il problema è lo stesso del notebook multi-round: 30 asset per impostazione predefinita. Nessun round di riduzione o solver MIS viene eseguito.

```bash
poetry install --with notebooks,experiments
poetry run python -m qreduce.experiments.portfolio_classical --run-id 001_classical
```

Usare `--asset-count 60` per un'istanza più grande e un nuovo `--run-id` per ogni pubblicazione. Per il confronto usare lo stesso numero di asset dell'esperimento multi-round. SCIP richiede PySCIPOpt; un backend non disponibile può produrre una failure normalizzata.

In [ ]:
from pathlib import Path

from qreduce.experiments import load_experiment_run
from qreduce.notebooks.presentation import display_vertical

repository_root = next(
    (directory for directory in (Path.cwd(), *Path.cwd().parents)
     if (directory / "pyproject.toml").is_file()),
    None,
)
if repository_root is None:
    raise RuntimeError("Run this notebook from inside the QReduce repository.")

run_directory = repository_root / "outputs/001_classical"
artifacts = load_experiment_run(run_directory)
if artifacts.manifest["experiment_id"] != "synthetic-portfolio-classical" or not artifacts.is_classical_only:
    raise ValueError("The selected artifact is not a classical-only portfolio run.")
result = artifacts.result

## 1. Configurazione, provenance e stato

In [ ]:
display_vertical(dict(artifacts.manifest), title="Manifest")
display_vertical(dict(artifacts.config["classical_solve"]), title="Solver classico")
display_vertical(dict(result["run_result"]), title="Risultato normalizzato")
display_vertical(dict(result["metrics"]), title="Metriche complessive")

## 2. Problema e dimensioni

Il solver opera sul problema originale, senza esclusioni euristiche.

In [ ]:
display_vertical({
    "asset_count": len(artifacts.problem["asset_ids"]),
    "original_variable_counts": result["original_variable_counts"],
    "residual_variable_counts": result["residual_variable_counts"],
    "round_count": artifacts.reduction_round_count,
    "recovery_attempt_count": len(artifacts.recovery_attempts),
}, title="Dimensioni")

## 3. Soluzione e verifica

La certificazione dell'ottimalità dipende dall'esito del solver e del post-check; un run può terminare senza soluzione valida.

In [ ]:
display_vertical({
    "final_retained_item_ids": result["final_retained_item_ids"],
    "cumulative_fixing_manifest": result["cumulative_fixing_manifest"],
    "classical_result": result["classical_result"],
    "postcheck_result": result["postcheck_result"],
    "original_problem_optimality_claim": result["original_problem_optimality_claim"],
}, title="Esito finale")